In [ ]:
import logging
from typing import Callable

logger = logging.getLogger("TEV")
logging.basicConfig(
    format="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
    filename= "output/count_gates.log",
    encoding="utf-8",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
)
logging.getLogger('crsq').setLevel(logging.INFO)

from crsq.blocks import (
    wave_function,
    hamiltonian,
    discretization,
    rfqhamiltonian,
    radial_func_qrom,
    hamiltonian2
)

from crsq.utils import circuit_tools as ctools

from qiskit import QuantumCircuit

dim = 3

def rfunc_for_rfq(r):
    return r

def count_gate_rfq(n):
    dq = 1 / 2 ** (n - 1)
    tb = radial_func_qrom.RadialFuncQrom(n, dq, rfunc_for_rfq, verbose=False)
    qc = ctools.decompose_circuit_to_privmities(tb.circuit)
    ops = qc.count_ops()
    ops2 = ctools.merge_controled_gates(ops)
    print(f"n:{n}  rfq count_ops:{ops2}")
    return qc

# count_gates_and_save_as_csv(5, 8, count_gate_rfq, "rfqrom")
def rfunc_for_rfqh(r):
    return 1/(r+0.5)

def print_summary(n: int, qc: QuantumCircuit, label: str):
    ops = qc.count_ops()
    ops2 = ctools.merge_controled_gates(ops)
    width = qc.width()
    print(f"{label} [{n}] width: {width}, count_ops:{ops2}")

def count_gate_arithmetic_elec_potential(n) -> QuantumCircuit:
    wfr_spec = wave_function.WaveFunctionRegisterSpec(
        dimension=dim,
        num_coordinate_bits=n,
        space_length=32,
        num_electrons=1,
        num_moving_nuclei=0,
        num_stationary_nuclei=1,
    )
    if dim == 2:
        pos = (0,0)
    elif dim == 3:
        pos = (0,0,0)
    ham_spec = hamiltonian.HamiltonianSpec(
        wfr_spec, nuclei_data=[{"charge": 1, "pos": pos}]
    )
    delta_t=1e-3
    disc_spec = discretization.DiscretizationSpec(delta_t)
    epb = hamiltonian.ElectronPotentialBlock(ham_spec, disc_spec)
    qc = ctools.decompose_circuit_to_privmities(epb.circuit)
    print_summary(n, qc, "Arithmetic Hamiltonian")
    return qc

def count_gate_rfq(n, label, use_symmetry, use_transpose, use_gray_code) -> QuantumCircuit:
    wfr_spec = wave_function.WaveFunctionRegisterSpec(
        dimension=dim,
        num_coordinate_bits=n,
        space_length=32,
        num_electrons=1,
        num_moving_nuclei=0,
        num_stationary_nuclei=1,
    )
    ham_spec = hamiltonian.HamiltonianSpec(
        wfr_spec, nuclei_data=[{"charge": 1, "pos": [0, 0]}]
    )
    rfunc = rfunc_for_rfqh
    rfq_spec = rfqhamiltonian.RfqPotentialSpec(
        wfr_spec, rfunc, rfunc,
        use_symmetry=use_symmetry,
        use_transpose=use_transpose,
        use_gray_code=use_gray_code)
    delta_t=1e-3
    disc_spec = discretization.DiscretizationSpec(delta_t)
    repb = rfqhamiltonian.RfqElectronPotentialBlock(rfq_spec, ham_spec, disc_spec)
    qc = ctools.decompose_circuit_to_privmities(repb.circuit)
    print_summary(n, qc, label)
    return qc

def count_gate_vsqrom_newton(n: int) -> QuantumCircuit:
    wfr_spec = wave_function.WaveFunctionRegisterSpec(
        dimension=dim,
        num_coordinate_bits=n,
        space_length=32,
        num_electrons=1,
        num_moving_nuclei=0,
        num_stationary_nuclei=1,
    )
    ham_spec = hamiltonian.HamiltonianSpec(
        wfr_spec, nuclei_data=[{"charge": 1, "pos": [0, 0]}]
    )
    ham2 = hamiltonian2.InverseSquareRoot(wfr_spec)
    qc = ctools.decompose_circuit_to_privmities(ham2.circuit)
    print_summary(n, qc, "VSQROM + Newton-Raphson")
    return qc


def count_gates_and_save_as_csv(N0, N, count_gate_func: Callable[[int], QuantumCircuit], label):
    outdir = "output/count_gates"
    filename = f"{outdir}/hamiltonian-gatecount-{label}-{N0}-{N}.csv"
    oplist = []
    for n in range(N0, N+1):
        qc = count_gate_func(n)
        width = qc.width()
        odict = qc.count_ops()
        odict2 = ctools.merge_controled_gates(odict)
        odict2["n"] = n
        odict2["width"] = width
        oplist.append(odict2)
    cols0 = [
        "n",
        "width",
        "mcx",
        "ccx",
        "cswap",
        "cx",
        "crz",
        "cp",
        "cu",
        "x",
        "h",
        "rz",
        "u",
        "p",
    ]
    cols = [col for col in cols0 if col in oplist[0]]
    with open(filename, "w") as f:
        f.write(",".join(cols) + "\n")
        for op in oplist:
            f.write(",".join([str(op[col]) for col in cols]) + "\n")

# count_gate_arithmetic_elec_potential(5)

nmin = 5
nmax = 9

# count_gates_and_save_as_csv(nmin, nmax, count_gate_vsqrom_newton, "vsqrom-newton")
# count_gates_and_save_as_csv(nmin, nmax, count_gate_arithmetic_elec_potential, "arith-elec-potential")

def count_gate_sawtooth_qrom(n) -> QuantumCircuit:
    return count_gate_rfq(n, "Sawtooth", False, False, False)

def count_gate_sym_qrom(n) -> QuantumCircuit:
    return count_gate_rfq(n, "Symmetry RFQROM", True, False, False)

def count_gate_sym_trans_qrom(n) -> QuantumCircuit:
    return count_gate_rfq(n, "Symmetry Transpose RFQROM", True, True, False)

def count_gate_gray_code_qrom(n) -> QuantumCircuit:
    return count_gate_rfq(n, "Gray code QROM", False, False, True)

count_gates_and_save_as_csv(nmin, nmax, count_gate_gray_code_qrom, "gray-code-qrom")
count_gates_and_save_as_csv(nmin, nmax, count_gate_sawtooth_qrom, "sawtooth-qrom")
count_gates_and_save_as_csv(nmin, nmax, count_gate_sym_qrom, "rfqrom-sym")
count_gates_and_save_as_csv(nmin, nmax, count_gate_sym_trans_qrom, "rfqrom-sym-trans")


Gray code QROM [5] width: 16, count_ops:{'cx': 1024, 'rz': 256}
Gray code QROM [6] width: 19, count_ops:{'cx': 4096, 'rz': 1024}
Gray code QROM [7] width: 22, count_ops:{'cx': 16384, 'rz': 4096}
Gray code QROM [8] width: 25, count_ops:{'cx': 65536, 'rz': 16350}
Gray code QROM [9] width: 28, count_ops:{'cx': 262144, 'rz': 65128}
Sawtooth [5] width: 25, count_ops:{'cp': 1024, 'x': 513, 'cx': 512, 'ccx': 510, 'barrier': 2}
Sawtooth [6] width: 30, count_ops:{'cp': 4096, 'x': 2049, 'cx': 2048, 'ccx': 2046, 'barrier': 2}
Sawtooth [7] width: 35, count_ops:{'cp': 16384, 'x': 8193, 'cx': 8192, 'ccx': 8190, 'barrier': 2}
Sawtooth [8] width: 40, count_ops:{'cp': 65536, 'x': 32769, 'cx': 32768, 'ccx': 32766, 'barrier': 2}
Sawtooth [9] width: 45, count_ops:{'cp': 262144, 'x': 131073, 'cx': 131072, 'ccx': 131070, 'barrier': 2}
Symmetry RFQROM [5] width: 31, count_ops:{'cx': 525, 'u': 336, 'ccx': 470, 'cp': 303, 'x': 163, 'cu': 48, 'p': 4, 'barrier': 2}
Symmetry RFQROM [6] width: 37, count_ops:{'cp':